In [37]:
import requests
import xml.etree.ElementTree as ET
import time

import sqlite3

In [38]:
esearch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

def get_ids(year):
    params = {
        "db": "pubmed", 
        "term": f"fibromyalgia AND {year}[pdat]", 
        "retmode": "json", 
        "retmax": "9999"
        }
    
    response = requests.get(esearch_url, params = params)
    time.sleep(0.4)
    data = response.json()

    return data["esearchresult"]["idlist"]

In [39]:
get_ids(2020)

['34840792',
 '34756320',
 '34387961',
 '34387956',
 '34222775',
 '34150583',
 '34055343',
 '34055335',
 '34046567',
 '34046566',
 '34020527',
 '33987505',
 '33842876',
 '33758815',
 '33758814',
 '33758807',
 '33758805',
 '33758802',
 '33739764',
 '33729353',
 '33601254',
 '33585505',
 '33583165',
 '33526114',
 '33521570',
 '33473332',
 '33467223',
 '33458664',
 '33458655',
 '33454042',
 '33437956',
 '33424538',
 '33415907',
 '33396651',
 '33396279',
 '33383293',
 '33380712',
 '33380711',
 '33371148',
 '33370398',
 '33370324',
 '33369515',
 '33367213',
 '33364942',
 '33364568',
 '33364564',
 '33363580',
 '33360825',
 '33355406',
 '33352747',
 '33343707',
 '33343483',
 '33342601',
 '33338002',
 '33337993',
 '33337990',
 '33337987',
 '33337931',
 '33331277',
 '33328761',
 '33327434',
 '33326048',
 '33321937',
 '33318554',
 '33315206',
 '33312004',
 '33303701',
 '33301346',
 '33293377',
 '33293261',
 '33291268',
 '33280616',
 '33275092',
 '33271010',
 '33266511',
 '33265969',
 '33259854',

In [40]:
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

def fetch_papers(ids):
    params = {
        "db": "pubmed", 
        "id": ",".join(ids), 
        "retmode": "xml"
        }
    
    response = requests.get(fetch_url, params = params)
    time.sleep(0.4)
    root = ET.fromstring(response.text)

    papers = []
    for article in root.findall(".//PubmedArticle"):
        pmid = article.findtext(".//PMID")
        title_el = article.find(".//ArticleTitle")
        title = "".join(title_el.itertext()) if title_el is not None else ""
        abstract = "\n".join("".join(p.itertext()) for p in article.findall(".//AbstractText"))
        date = article.find(".//PubDate")
        year = date.findtext("Year")
        pub_types = [pt.text for pt in article.findall(".//PublicationType")]
        study_types = ",".join(pub_types)

        papers.append({"pmid": pmid, "title": title, "abstract": abstract, "year": year, "types": study_types})
    return papers

In [ ]:
conn = sqlite3.connect("data/fibro.db")
conn.execute("""
    CREATE TABLE IF NOT EXISTS papers (
        pmid TEXT PRIMARY KEY,
        title TEXT,
        abstract TEXT,
        year TEXT,
        types TEXT
    )
""")

In [42]:
for year in range(1970, 2027):
    ids = get_ids(year)
    print(f"{year}: {len(ids)} papers")

    for start in range(0, len(ids), 200):
        batch = ids[start:start + 200]
        papers = fetch_papers(batch)
        conn.executemany(
            "INSERT OR IGNORE INTO papers (pmid, title, abstract, year, types) VALUES (:pmid, :title, :abstract, :year, :types)",
            papers
        )
        conn.commit()

1970: 16 papers
1971: 25 papers
1972: 24 papers
1973: 26 papers
1974: 22 papers
1975: 20 papers
1976: 18 papers
1977: 16 papers
1978: 10 papers
1979: 14 papers
1980: 8 papers
1981: 17 papers
1982: 21 papers
1983: 16 papers
1984: 23 papers
1985: 20 papers
1986: 57 papers
1987: 27 papers
1988: 59 papers
1989: 120 papers
1990: 96 papers
1991: 83 papers
1992: 104 papers
1993: 119 papers
1994: 152 papers
1995: 156 papers
1996: 162 papers
1997: 206 papers
1998: 236 papers
1999: 210 papers
2000: 255 papers
2001: 241 papers
2002: 295 papers
2003: 261 papers
2004: 341 papers
2005: 297 papers
2006: 363 papers
2007: 408 papers
2008: 489 papers
2009: 566 papers
2010: 556 papers
2011: 644 papers
2012: 630 papers
2013: 644 papers
2014: 648 papers
2015: 685 papers
2016: 686 papers
2017: 699 papers
2018: 652 papers
2019: 704 papers
2020: 826 papers
2021: 891 papers
2022: 895 papers
2023: 964 papers
2024: 977 papers
2025: 1035 papers
2026: 1050 papers
